# Markov chains — companion notebook

Runnable companion for [`markov-chains.md`](./markov-chains.md).

1. Build a 3-state chain and compute its stationary distribution two ways: solving $\boldsymbol\pi P = \boldsymbol\pi$ and taking $P^n$ for large $n$.
2. Simulate trajectories; show the empirical visit frequencies match $\boldsymbol\pi$ (ergodic theorem).
3. Visualize convergence of $\boldsymbol\pi_t$ from different initial states.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

# 3-state row-stochastic transition matrix
P = np.array([
    [0.7, 0.2, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5],
])
assert np.allclose(P.sum(axis=1), 1)
n_states = P.shape[0]

## 1. Stationary distribution

Solve $\boldsymbol\pi P = \boldsymbol\pi$ via the left eigenvector of eigenvalue $1$, and cross-check with $P^n$ for large $n$.

In [ ]:
# Left eigenvectors == right eigenvectors of P^T
vals, vecs = np.linalg.eig(P.T)
# pick the eigenvector for eigenvalue closest to 1
idx = np.argmin(np.abs(vals - 1.0))
pi_eig = np.real(vecs[:, idx])
pi_eig = pi_eig / pi_eig.sum()
print('Stationary pi (eigenvector) =', pi_eig)

# Power iteration: P^n -> rows all equal to pi
Pn = np.linalg.matrix_power(P, 200)
print('\nP^200 (every row -> pi):')
print(Pn)

print('\nAll eigenvalues of P:', sorted(np.abs(vals).round(4), reverse=True),
      '   spectral gap =', round(1 - sorted(np.abs(vals))[-2], 4))

## 2. Simulation and ergodic theorem

In [ ]:
def simulate(P, n_steps, x0=0, rng=rng):
    xs = np.empty(n_steps, dtype=int)
    x = x0
    for t in range(n_steps):
        xs[t] = x
        x = rng.choice(P.shape[0], p=P[x])
    return xs

n_steps = 20_000
path = simulate(P, n_steps, x0=0)

# Empirical visit frequencies
freq = np.bincount(path, minlength=n_states) / n_steps
print('Empirical frequencies =', freq)
print('Stationary pi         =', pi_eig)

## 3. Convergence of $\boldsymbol\pi_t$ from different initial states

Plot $\boldsymbol\pi_t = \mathbf{e}_i P^t$ for each starting state $i$; all three curves collapse onto $\boldsymbol\pi$.

In [ ]:
T = 25
fig, axes = plt.subplots(1, n_states + 1, figsize=(4 * (n_states + 1), 4))

for start, ax in zip(range(n_states), axes[:-1]):
    pi_t = np.zeros((T, n_states))
    pi = np.zeros(n_states); pi[start] = 1.0
    Pi = pi.copy()
    for t in range(T):
        pi_t[t] = Pi
        Pi = Pi @ P
    for s in range(n_states):
        ax.plot(pi_t[:, s], marker='o', ms=3, label=f'state {s}')
        ax.axhline(pi_eig[s], color=f'C{s}', ls=':', alpha=0.5)
    ax.set_title(f'start in state {start}')
    ax.set_xlabel('t'); ax.set_ylabel('Pr(X_t = s)')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Bar chart of stationary distribution
ax = axes[-1]
ax.bar(range(n_states), pi_eig, color='tab:gray', alpha=0.7, label='stationary pi')
ax.bar(range(n_states), freq, color='tab:orange', alpha=0.5, label=f'empirical (n={n_steps})')
ax.set_xticks(range(n_states))
ax.set_title('Stationary vs empirical')
ax.set_xlabel('state'); ax.set_ylabel('probability')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Pitfall: a periodic chain

A chain that alternates strictly between two states (period 2) does not satisfy $P^n \to \boldsymbol{1}\boldsymbol\pi^\top$; the marginal oscillates. Adding a self-loop (a *lazy* chain) restores aperiodicity.

In [ ]:
P_periodic = np.array([[0.0, 1.0],
                       [1.0, 0.0]])
print('Periodic chain: powers oscillate')
print('P^1 =', P_periodic.flatten())
print('P^2 =', np.linalg.matrix_power(P_periodic, 2).flatten())
print('P^3 =', np.linalg.matrix_power(P_periodic, 3).flatten())

P_lazy = 0.5 * np.eye(2) + 0.5 * P_periodic  # add self-loops
print('\nLazy version converges to uniform [0.5, 0.5]:')
print('P_lazy^50 =\n', np.linalg.matrix_power(P_lazy, 50))